In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualização gráfica inline no Jupyter notebook
%matplotlib inline

# Como avalio a regressão do fator p (*p-factor*) no nível do participante?

Estime o fator p observado do HBN a partir do EEG em repouso usando uma linha por participante.
Seis participantes do R5 mini mantêm a aquisição delimitada. Essas gravações são as
derivadas do desafio a 100 Hz, já filtradas em 0.5–50 Hz. O primeiro uso baixa seis gravações;
o tamanho exato depende de sua duração. Este pequeno exercício com validação deixando um participante
de fora (*leave-one-participant-out*) não é uma estimativa do placar (*leaderboard*) do desafio.


## Antes de começar

Use um ambiente com EEGDash instalado juntamente com MNE, NumPy, scikit-learn e
Matplotlib. Não é necessária GPU. Defina ``EEGDASH_CACHE_DIR`` para reutilizar as seis
gravações em estado de repouso; o subconjunto requer aproximadamente 100 MB no primeiro download.
Cortar o sinal (*crop*) reduz o tempo de processamento e memória, mas não a quantidade de bytes necessária para
adquirir cada gravação.

O fator p é um fenótipo observado no nível do participante fornecido pelo
desafio. Não é um rótulo de ensaio, um diagnóstico feito a partir de EEG ou um valor a ser
reconstruído a partir do identificador de um participante. Um sujeito contribui com exatamente uma
linha de características e um alvo, de modo que gravações longas não podem aumentar o peso desse sujeito
simplesmente por gerarem mais janelas.



In [ ]:
# Importa módulos de sistema operacional e manipulação de diretórios
import os
from pathlib import Path

# Importa bibliotecas para plotagem e manipulação de matrizes numéricas
import matplotlib.pyplot as plt
import numpy as np
# Importa regressor dummy, Ridge, métricas de regressão, LOO, pipeline e escalonador do scikit-learn
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset do desafio, funções espectrais e mapa da versão mini do EEGDash
from eegdash import EEGChallengeDataset
from eegdash.features import spectral_preprocessor, spectral_bands_power
from eegdash.const import SUBJECT_MINI_RELEASE_MAP

Carregar alvos observados dos participantes e voltagens gravadas.
%%
Carregar alvos reais antes de calcular características
-----------------------------------------------------

A lista ordenada da versão mini fornece um subconjunto pequeno e reprodutível em vez
de selecionar participantes com base em seus resultados. ``target_name`` nomeia o fenótipo
observado; ``description_fields`` torna a identidade e o alvo disponíveis juntamente com cada
gravação. Os metadados impressos permitem verificar a união entre sinal e participante.

Uma gravação ausente falha na verificação de cobertura. Alvos ausentes ou não numéricos
devem ser investigados na fonte em vez de preenchidos com a média do grupo ou com um número
aleatório. Para uma coorte maior, especifique exclusões por alvos ausentes antes de ajustar
um modelo e relate o número resultante de pessoas.



In [ ]:
# Seleciona os primeiros 6 sujeitos da lista ordenada da versão mini R5
subjects = sorted(SUBJECT_MINI_RELEASE_MAP["R5"])[:6]
# Carrega gravações de repouso contendo o p_factor nos metadados
dataset = EEGChallengeDataset(
    release="R5",
    mini=True,
    task="RestingState",
    subject=subjects,
    cache_dir=Path(
        os.environ.get("EEGDASH_CACHE_DIR", "~/.eegdash_cache")
    ).expanduser(),
    description_fields=["subject", "task", "p_factor"],
    target_name="p_factor",
)
# Exibe metadados dos 6 sujeitos selecionados
print(dataset.description.to_string(index=False))
# Assegura que todos os 6 sujeitos foram carregados com sucesso
assert len(dataset.datasets) == len(subjects)

## Resumir um intervalo fixo de repouso

``crop(tmax=59)`` retém a gravação do tempo zero até 59 segundos.
Esse horizonte fixo impede que diferenças de duração decidam quanto EEG contribui
para cada linha. Não isola uma condição ocular uniforme: as instruções de repouso gravadas
podem ocorrer dentro do intervalo.

Os espectros de Welch do EEGDash usam segmentos Hamming não sobrepostos de 200 amostras na
taxa original de 100 Hz, fornecendo espaçamento em frequência de 0.5 Hz. As características usam
quatro faixas: 1–4, 4–8, 8–13 e 13–30 Hz. O ``spectral_preprocessor`` do EEGDash calcula a PSD
e ``spectral_bands_power`` soma os bins selecionados. Multiplicar pelo espaçamento de 0.5 Hz
aproxima a potência da banda em V² antes de aplicar ``log10``. Cada banda inclui seu limite inferior
e exclui seu limite superior, portanto bandas adjacentes não compartilham bins. Todos os participantes
usam essa convenção. O pequeno piso (*floor*) numérico evita logaritmos indefinidos para canais planos,
como a referência de origem; ele não transforma um eletrodo plano em uma característica informativa.

A concatenação é organizada por bandas, retendo a ordem dos canais dentro de cada banda. A
asserção da ordem dos canais mantém as colunas de características comparáveis entre gravações.



In [ ]:
# Inicializa listas para armazenar características, alvos e identidades dos participantes
features, targets, identities = [], [], []
channels = None
# Itera por cada gravação para processamento do sinal contínuo
for recording in dataset.datasets:
    # Copia o sinal bruto, seleciona canais de EEG, corta os primeiros 60s (0 a 59s) e carrega em memória
    raw = recording.raw.copy().pick("eeg").crop(tmax=59).load_data()
    # Registra a lista de canais da primeira gravação como referência padrão
    channels = raw.ch_names if channels is None else channels
    # Garante consistência dos nomes e ordem de canais em todas as gravações
    assert raw.ch_names == channels
    # Calcula a densidade espectral de potência (PSD) de Welch com segmentos Hamming de 200 amostras a 100 Hz
    frequencies, psd = spectral_preprocessor(
        raw.get_data(),
        _metadata={"info": raw.info},
        f_min=1,
        f_max=30,
        nperseg=200,
        noverlap=0,
        window="hamming",
    )
    # Calcula a potência nas quatro bandas fisiológicas clássicas
    powers = spectral_bands_power(
        frequencies,
        psd,
        bands={"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)},
    )
    # Concatena os valores de potência das bandas e multiplica pela resolução em frequência (0.5 Hz)
    band_power = np.concatenate(list(powers.values())) * (
        frequencies[1] - frequencies[0]
    )
    # Aplica transformação logarítmica log10 com piso de 1e-30 para estabilidade numérica
    features.append(np.log10(np.maximum(band_power, 1e-30)))
    # Extrai o valor do fenótipo p_factor como float
    targets.append(float(recording.description["p_factor"]))
    # Registra o identificador do sujeito
    identities.append(str(recording.description["subject"]))
    # Exibe informações do sujeito processado
    print(
        identities[-1],
        len(raw.ch_names),
        raw.info["sfreq"],
        raw.annotations.description[:8],
    )

## Verificar a matriz de planejamento no nível do participante

``X`` tem formato ``(participantes, quatro bandas × canais)`` e ``y`` possui
um fator p observado por linha. Com as gravações atuais de 129 canais, isso
significa 516 preditores para apenas seis participantes. As asserções de identidade
e finitude detectam pessoas duplicadas, fenótipos ausentes e características inválidas.
Elas não testam se o EEG contém informações preditivas.

Esse cenário de altíssima dimensionalidade e amostra diminuta motiva regularização, mas nenhuma
penalização torna seis participantes suficientes para inferência clínica. A página
demonstra uma fronteira de avaliação independente do sujeito; não estabelece que as
características resultantes sejam invariantes à identidade do sujeito.



In [ ]:
# Converte listas de características e alvos em arrays numpy
X, y = np.asarray(features), np.asarray(targets)
# Valida que cada linha pertence a um sujeito único e que todos os valores de X e y são finitos
assert len(set(identities)) == len(y) and np.isfinite(X).all() and np.isfinite(y).all()
# Exibe o formato da matriz de características (6 sujeitos, 516 colunas) e os alvos observados
print("Participant features:", X.shape, "observed targets:", y)

Todo escalonamento e ajuste da linha de base ocorrem dentro da partição do participante retido.
%%
Ajustar dentro de cada partição de participante retido
------------------------------------------------------

O leave-one-out ajusta seis modelos, cada um usando cinco pessoas e prevendo a
pessoa restante uma vez. O ``StandardScaler`` é reajustado dentro de cada pipeline para que
suas médias e dispersões nunca incluam a linha retida. O ``alpha=10`` do Ridge é uma
penalidade ilustrativa fixa, não um valor selecionado a partir desses seis erros de teste.

O modelo dummy recalcula separadamente a média do treino em cada partição. O erro médio
absoluto é relatado na escala fornecida do fator p, com peso igual por pessoa.
Um erro do Ridge menor do que o erro do dummy seria uma evidência descritiva apenas neste
subconjunto; um maior é igualmente legítimo. A diagonal no gráfico de dispersão denota predição
exata, não uma linha de regressão ajustada.



In [ ]:
# Inicializa arrays para armazenar predições do Ridge e do modelo base dummy
predicted, baseline = np.empty_like(y), np.empty_like(y)
# Validação cruzada deixando um participante de fora (LOOCV)
for train, test in LeaveOneOut().split(X):
    # Garante que os sujeitos de treino e teste são estritamente disjuntos
    assert set(np.asarray(identities)[train]).isdisjoint(np.asarray(identities)[test])
    # Constrói pipeline com padronizador e regressão Ridge (alpha=10)
    model = make_pipeline(StandardScaler(), Ridge(alpha=10))
    # Ajusta o modelo e realiza predição para o participante mantido fora
    predicted[test] = model.fit(X[train], y[train]).predict(X[test])
    # Ajusta o modelo dummy (média do treino) e gera predição basal
    baseline[test] = DummyRegressor().fit(X[train], y[train]).predict(X[test])
# Exibe os erros absolutos médios (MAE) dos modelos
print("Participant MAE:", mean_absolute_error(y, predicted))
print("Training-mean MAE:", mean_absolute_error(y, baseline))
# Gera gráfico de dispersão: valores observados versus valores previstos de p_factor
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(y, predicted, label="held-out participant")
# Traça linha diagonal pontilhada de predição perfeita (y = x)
ax.plot([y.min(), y.max()], [y.min(), y.max()], "k--")
ax.set(xlabel="Observed p-factor", ylabel="Predicted p-factor")
ax.legend()
# Exibe o gráfico
plt.show()

## Separar desenvolvimento de modelo de interpretação clínica

Primeiro aumente o número de participantes retidos de forma independente. Se você
desejar escolher a penalidade ridge, canais, bandas ou intervalo de repouso, faça-o
usando uma divisão interna de validação apenas no treino e mantenha a partição externa
de participantes intocada. Reutilizar esses erros externos para selecionar características
os torna resultados de desenvolvimento de modelo em vez de uma avaliação final.

Uma extensão útil é comparar as características de EEG observadas com uma linha de base
pré-especificada apenas com metadados, usando as mesmas pessoas e partições. Relate exclusões
por alvos ausentes e incerteza no nível do participante. Evite interpretar um coeficiente
ajustado como biomarcador clínico quando os preditores são correlacionados e a amostra é tão pequena.

Exemplo relacionado de avaliação: [Braindecode train, test and tune](https://braindecode.org/stable/auto_examples/model_building/plot_how_train_test_and_tune.html).

